# Explainable AI for Cryptographic Behaviour Classification
## Phase 1 — Data Generation

In [ ]:
from data_generation import generate_dataset
import pandas as pd

df = generate_dataset()
print(df['label'].value_counts())

## Phase 2 — Feature Extraction

In [ ]:
import ast
import pandas as pd
from feature_extraction import extract_features

raw_df = pd.read_csv('data/dataset.csv')

records = [
    {**extract_features(ast.literal_eval(row.bytes)), "label": row.label}
    for row in raw_df.itertuples(index=False)
]

feature_df = pd.DataFrame(records)
feature_df.to_csv('data/features.csv', index=False)
print(feature_df.head())
print(feature_df['label'].value_counts())

## Sanity Check — Feature Separation by Class

In [ ]:
feature_df.groupby('label')[['entropy', 'chi_square', 'unique_byte_ratio']].mean()

## Phase 3 — Model Training & Evaluation

In [ ]:
from model_training import train_and_evaluate
import pandas as pd

feature_df = pd.read_csv('data/features.csv')
model, X_train, X_test, y_train, y_test = train_and_evaluate(feature_df)

## Phase 4 — XAI with SHAP

In [ ]:
from xai_explanation import run_shap_analysis

run_shap_analysis(model, X_train, X_test, y_test)

## Discussion

### Key Findings

1. **Shannon entropy is the dominant feature** for separating plaintext from ciphertext. The SHAP summary plot confirms that entropy alone achieves near-perfect separation — low entropy (~4-6) indicates plaintext, while high entropy (~7.2) indicates ciphertext.

2. **AES and DES are the hardest pair to distinguish.** Both produce near-uniform byte distributions with entropy ~7.2 and chi-square ~255. The model distinguishes them primarily through structural artifacts (IV length: 16 bytes for AES vs 8 bytes for DES) rather than fundamental cryptographic properties.

3. **RSA is moderately separable.** Its fixed 256-byte block structure and OAEP padding create subtle statistical differences from AES/DES, yielding 91% recall.

4. **Plaintext has the lowest recall (77%)** because structured plaintext with binary components can produce entropy values overlapping with RSA ciphertext.

5. **SHAP waterfall plots provide instance-level transparency.** For any individual prediction, we can trace exactly which features contributed and by how much — essential for building trust in security applications.

### Limitations

- Synthetic data does not capture real-world TLS behavior, protocol headers, or network artifacts
- AES/DES distinction relies on IV length differences that may not persist in real traffic
- Feature set could be extended with autocorrelation, spectral analysis, and n-gram statistics
- Model trained on fixed key sizes; variation may change the statistical fingerprint

### Future Work

- Apply to real network traffic datasets (CICIDS-2017, UNSW-NB15)
- Include additional protocols (ChaCha20, 3DES, Blowfish)
- Investigate adversarial robustness against byte-level perturbations